<a href="https://colab.research.google.com/github/muhallilahnaf/RAGDSS/blob/master/RAGDSS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Github repo link
# https://github.com/muhallilahnaf/RAGDSS

In [ ]:
## in terminal
## apt-get install zstd
## curl -fsSL https://ollama.com/install.sh | sh
## ollama serve &
## ollama run llama3.2, sqlcode:7b

In [ ]:
!pip install ollama
!pip install chromadb
!pip install sqlglot
!pip install openrouter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentel

In [ ]:
!wget https://raw.githubusercontent.com/muhallilahnaf/RAGDSS/refs/heads/master/prices_clean.csv
!wget https://raw.githubusercontent.com/muhallilahnaf/RAGDSS/refs/heads/master/reviews_clean.csv

--2026-05-01 08:40:39--  https://raw.githubusercontent.com/muhallilahnaf/RAGDSS/refs/heads/master/prices_clean.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 44971 (44K) [text/plain]
Saving to: ‘prices_clean.csv’

prices_clean.csv    100%[===================>]  43.92K  --.-KB/s    in 0s      

2026-05-01 08:40:40 (139 MB/s) - ‘prices_clean.csv’ saved [44971/44971]

--2026-05-01 08:40:40--  https://raw.githubusercontent.com/muhallilahnaf/RAGDSS/refs/heads/master/reviews_clean.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length:

In [ ]:
import sqlite3
import uuid
import pandas as pd
import ollama
import json
import chromadb
import sqlglot
from sqlglot.optimizer.qualify import qualify
from openrouter import OpenRouter

from google.colab import userdata
apikey = userdata.get('openrouter2')

# test to check if llm is responding
client = OpenRouter(api_key=apikey)
# response = client.chat.send(
#   model="openai/gpt-oss-20b:free",
#   messages=[
#     {"role": "user", "content": "What is the meaning of life?"}
#   ],
# )

# print(response.choices[0].message.content)

In [ ]:
# test to check if llm is responding
# response = ollama.chat(model='llama3.2', messages=[
#   {'role': 'user', 'content': 'Why is the sky blue?'},
# ])
# print(response['message']['content'])

In [ ]:
# start a database connection
conn = sqlite3.connect('RAGDSS.db')
cursor = conn.cursor()

# read prices csv
df = pd.read_csv('prices_clean.csv')
df.head()

,id,brand,categories,dateAdded,dateUpdated,name,currency,amount,date,isSale,merchant,shipping,condition
0,AV000tWuGV-KLJ3ac2-b,Amazon,Amazon Devices,2017-07-12T03:23:38Z,2017-07-25T02:04:06Z,Echo Show - Black,USD,229.99,2017-07-21T23:11:11.925Z,False,Amazon.com,FREE Shipping.,NaN
1,AV00l7jV-jtxr-f30lnX,Amazon,Amazon Devices,2017-07-12T02:19:04Z,2017-08-13T08:29:10Z,Amazon Echo Dot Case (fits Echo Dot 2nd Genera...,USD,19.99,2017-07-26T00:00:12.394Z,False,Amazon.com,FREE Shipping on orders over USD 25.00,NaN
2,AV00lzP7GV-KLJ3ac0uk,Amazon,Amazon Devices,2017-07-12T02:18:30Z,2017-08-13T08:28:52Z,Amazon Echo Dot Case (fits Echo Dot 2nd Genera...,USD,14.99,2017-07-25T23:52:21.319Z,False,Amazon.com,FREE Shipping on orders over USD 25.00,NaN
3,AV00lzP7GV-KLJ3ac0uk,Amazon,Amazon Devices,2017-07-12T02:18:30Z,2017-08-13T08:28:52Z,Amazon Echo Dot Case (fits Echo Dot 2nd Genera...,USD,9.99,2017-07-25T23:52:21.319Z,True,Amazon.com,FREE Shipping on orders over USD 25.00,NaN
4,AV00lzd5GV-KLJ3ac0ul,Amazon,Amazon Devices,2017-07-12T02:18:31Z,2017-07-18T23:46:14Z,All-New Amazon Fire TV Game Controller,USD,49.99,2017-06-30T15:54:25.350Z,False,easy-to-open packaging,FREE Shipping.,NaN


In [ ]:
filter_table_name = 'prices'

# add prices data to database
df.to_sql(filter_table_name, conn)

# test to check database works
# cursor.execute("SELECT * FROM prices LIMIT 1")
# print(cursor.fetchall())

# get table description
cursor.execute("PRAGMA table_info(prices)")
columns = cursor.fetchall()
columns_info = [(col[1], col[2]) for col in columns]
table_description = "Columns:\n" + "\n".join([f"  - {name}: {col_type}" for name, col_type in columns_info])
print(table_description)

Columns:
  - index: INTEGER
  - id: TEXT
  - brand: TEXT
  - categories: TEXT
  - dateAdded: TEXT
  - dateUpdated: TEXT
  - name: TEXT
  - currency: TEXT
  - amount: REAL
  - date: TEXT
  - isSale: INTEGER
  - merchant: TEXT
  - shipping: TEXT
  - condition: TEXT


In [ ]:
question = input('Ask a business question: ')

Ask a business question: Compare the reviews of Kindle Fire HDX 8.9" that are priced below $400 and on sale


In [ ]:
json_format = """
{
  "filters": {
    "column_name1": "column_value1",
    "column_name2": "column_value2"
  },
  "semantic_intent": "...",
  "original_query": "..."
}
"""

json_output_example = """
{
  "filters": {
    "brand": "Samsung",
    "amount": {"operator": "<", "value": 500}
  },
  "semantic_intent": "battery issues, battery complaints",
  "original_query": "Which Samsung phones under $500 have battery issues?"
}
"""

initial_context = """
You are an information extraction system.

Your task is to analyze a business question and extract:
1. Structured filters (ONLY for the {filter_table_name} table)
2. Semantic intent (for review search)

Schema of {filter_table_name}:
{table_description}

Rules:
- Only extract filters that map directly to the schema
- Do NOT infer values not explicitly mentioned
- Keep semantic intent focused on qualitative aspects (e.g., battery, durability, complaints)
- If a filter is not present, omit it

Return STRICT JSON:

{json_format}

Example:

Query:
"Which Samsung phones under $500 have battery issues?"

Output:
{json_output_example}

Now process:
{question}
""".format(
    filter_table_name=filter_table_name,
    table_description=table_description,
    question=question,
    json_format=json_format,
    json_output_example=json_output_example
)

response = client.chat.send(
    model="openai/gpt-oss-20b:free",
    messages=[
      {"role": "user", "content": initial_context}
    ],
  )

filter_json = response.choices[0].message.content
print(filter_json)

{
  "filters": {
    "name": "Kindle Fire HDX 8.9",
    "amount": { "operator": "<", "value": 400 },
    "isSale": 1
  },
  "semantic_intent": "performance issues, battery life complaints, display quality, durability concerns",
  "original_query": "Compare the reviews of Kindle Fire HDX 8.9\" that are priced below $400 and on sale"
}


In [ ]:
# generate sql using llm

sql_json_example = """
"filters": {
    "brand": "Samsung",
    "amount": {"operator": "<", "value": 500}
}
"""

sql_context = """
You are a SQL generation system.

Generate a SQL query to retrieve product IDs from the {filter_table_name} table.

Schema of {filter_table_name}:
{table_description}

Rules:
- ONLY generate a SELECT query
- ONLY return the column: id
- DO NOT include explanations
- DO NOT hallucinate columns
- Use WHERE clauses only if filters exist
- Use correct SQL syntax

Input JSON:
{filter_json}

Output:
SQL query only.

Example:

Input:
{sql_json_example}

Output:
SELECT id FROM products
WHERE brand = 'Samsung' AND price < 500;

Now generate SQL:
""".format(
    filter_table_name=filter_table_name,
    table_description=table_description,
    filter_json=filter_json,
    sql_json_example=sql_json_example
)

response = client.chat.send(
  model="openai/gpt-oss-20b:free",
  messages=[
    {"role": "user", "content": sql_context}
  ],
)

llm_sql = response.choices[0].message.content
print(llm_sql)

SELECT id FROM prices
WHERE name = 'Kindle Fire HDX 8.9' AND amount < 400 AND isSale = 1;


In [ ]:
llm_sql = "SELECT id FROM prices WHERE name = 'Kindle Fire HDX 8.9\"' AND amount < 400 AND isSale = 1;"

In [ ]:
# validate sql
q = ''
for name, col_type in columns_info:
  q += f'"{name}": "{col_type}",'
# remove last comma
q = q[:-1]
schema = '{' + f'"{filter_table_name}"' + ': {' + q + '}}'
# print(schema)
schema = json.loads(schema)
expression = sqlglot.parse_one(llm_sql, read="sqlite")
qualified_expression = qualify(expression, schema=schema, dialect="sqlite")

print('validated sql')
print(qualified_expression.sql(dialect="sqlite"))

# execute validated sql
cursor.execute(qualified_expression.sql(dialect="sqlite"))
rows = cursor.fetchall()
print(rows)

validated sql
SELECT "prices"."id" AS "id" FROM "prices" AS "prices" WHERE "prices"."name" = 'Kindle Fire HDX 8.9"' AND "prices"."amount" < 400 AND "prices"."issale" = 1
[('AVpfpzCi1cnluZ0-oxEr',), ('AVpfpzCi1cnluZ0-oxEr',), ('AVpfpzCi1cnluZ0-oxEr',), ('AVpfpzCi1cnluZ0-oxEr',), ('AVpfpzCi1cnluZ0-oxEr',)]


In [ ]:
# read reviews data
dfr = pd.read_csv('reviews_clean.csv')
dfr.head()

,pid,reviews.date,reviews.doRecommend,reviews.numHelpful,reviews.rating,reviews.text,reviews.title,reviews.username
0,AVpe7AsMilAPnD_xQ78G,2015-08-08T00:00:00.000Z,NaN,139.0,5.0,I initially had trouble deciding between the p...,"Paperwhite voyage, no regrets!",Cristina M
1,AVpe7AsMilAPnD_xQ78G,2015-09-01T00:00:00.000Z,NaN,126.0,5.0,Allow me to preface this with a little history...,One Simply Could Not Ask For More,Ricky
2,AVpe7AsMilAPnD_xQ78G,2015-07-20T00:00:00.000Z,NaN,69.0,4.0,I am enjoying it so far. Great for reading. Ha...,Great for those that just want an e-reader,Tedd Gardiner
3,AVpe7AsMilAPnD_xQ78G,2017-06-16T00:00:00.000Z,NaN,2.0,5.0,I bought one of the first Paperwhites and have...,Love / Hate relationship,Dougal
4,AVpe7AsMilAPnD_xQ78G,2016-08-11T00:00:00.000Z,NaN,17.0,5.0,I have to say upfront - I don't like coroporat...,I LOVE IT,Miljan David Tanic


In [ ]:
# fetch rows with target product id
dfr_target = dfr[dfr['pid']==rows[0][0]]
display(dfr_target)

# vectorize review data
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="reviews")
collection.add(
    ids=[str(uuid.uuid4()) for _ in range(len(dfr_target))],
    documents=dfr_target['reviews.text'].tolist(),
    metadatas=dfr_target[[
        'reviews.date',
        'reviews.doRecommend',
        'reviews.numHelpful',
        'reviews.rating',
        'reviews.title',
        'reviews.username'
    ]].to_dict(orient='records')
)

,pid,reviews.date,reviews.doRecommend,reviews.numHelpful,reviews.rating,reviews.text,reviews.title,reviews.username
106,AVpfpzCi1cnluZ0-oxEr,NaN,NaN,402.0,3.0,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,B. Tarbuck
107,AVpfpzCi1cnluZ0-oxEr,2013-11-07T00:00:00.000Z,NaN,330.0,4.0,"Simply put, this is the Amazons iPad Air Kille...",An enlarged version of the 7 HDX - Same good i...,Amazon Reviewer
108,AVpfpzCi1cnluZ0-oxEr,2015-03-15T00:00:00.000Z,NaN,85.0,1.0,"After 15 months, my 500+ tablet is no longer u...",Dead after 15 months,Samuel Ruiz
109,AVpfpzCi1cnluZ0-oxEr,2014-10-30T00:00:00.000Z,NaN,59.0,3.0,Cons:1. Built on Android does not mean you can...,Not everything I expected...,Tina L. Harmon
110,AVpfpzCi1cnluZ0-oxEr,2013-11-07T00:00:00Z,NaN,NaN,NaN,"Simply put, this is the Amazons iPad Air Kille...",An enlarged version of the 7 HDX - Same good i...,Amazon Reviewer
111,AVpfpzCi1cnluZ0-oxEr,2013-11-07T00:00:00Z,NaN,NaN,NaN,"Simply put, this is the Amazons iPad Air Kille...",An enlarged version of the 7 HDX - Same good i...,Amazon Reviewer
112,AVpfpzCi1cnluZ0-oxEr,NaN,NaN,NaN,NaN,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,B. Tarbuck
113,AVpfpzCi1cnluZ0-oxEr,NaN,NaN,NaN,3.0,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,B. Tarbuck
114,AVpfpzCi1cnluZ0-oxEr,2013-11-07T00:00:00Z,NaN,NaN,NaN,"Simply put, this is the Amazons iPad Air Kille...",An enlarged version of the 7 HDX - Same good i...,Amazon Reviewer
115,AVpfpzCi1cnluZ0-oxEr,NaN,NaN,NaN,3.0,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,B. Tarbuck


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:06<00:00, 12.2MiB/s]


In [ ]:
vector_retrieval_context = """
You are a query rewriting system for semantic search.

Your task:
- Remove all structured filtering conditions already handled by SQL
- Keep ONLY the part relevant for semantic similarity search in reviews
- Expand slightly for better retrieval (add synonyms if useful)
- Keep it concise (1 short sentence or phrase)

Rules:
- Do NOT include brand, price, color, year, etc. if already used as filters
- Focus on user intent (issues, satisfaction, performance, complaints)
- If no semantic intent exists, return: "general product feedback"

Input:
Original Query: {question}
Filters Applied:
{filter_json}

Output:
Rewritten semantic query only.

Example:

Input:
Original Query: "Which Samsung phones under $500 have battery issues?"
Filters:
{sql_json_example}

Output:
"battery issues, battery complaints, poor battery life"

---

Now rewrite:
""".format(
    question=question,
    filter_json=filter_json,
    sql_json_example=sql_json_example
)

response = client.chat.send(
    model="openai/gpt-oss-20b:free",
    messages=[
      {"role": "user", "content": vector_retrieval_context}
    ],
  )

semantic_question = response.choices[0].message.content
print(semantic_question)

performance issues, battery life complaints, display quality, durability concerns


In [ ]:
results = collection.query(
    query_texts=[semantic_question],
    n_results=5
)
print(results)

{'ids': [['3513dd3e-c419-4288-bfda-ea09a31be0ce', '91bbe69f-abbf-405e-90ff-b199fd633b27', '36c06dbd-04f4-44f3-a03a-72bdd29b3283', '34cb7421-4582-4b67-9f78-ce3a42c523e0', '7d4a03e7-824d-4b70-b7d5-6e76808bb50c']], 'embeddings': None, 'documents': [['If you like the Kindle Fire HD then you will love the Kindle Fire HDX. The new device runs three times faster, is lighter in weight, and the battery lasts slightly longer. The sound quality on this device is excellent and the display quality is absolutely stunning. Below are some of the features to expect from the Fire HDX. BUILD QUALITY---There is NO blue haze on the 8.9 HDX.---The build quality is excellent.---Auto contrast is good (not buggy).---Operating system is very intuitive except after downloading free prime videos to watch offline. If you remove the videos from your Carousel then, from the video menu, you must go to the left side bar, and click *downloads* to see your videos. PRIME DOWNLOAD ISSUES (free video download limitations)P

In [ ]:
rag_context = """
You are a business analyst.

Answer the user's question using ONLY the provided review data.

Rules:
- Base your answer strictly on the reviews
- Ensure to answer the points mentioned in the semantic query
- Do NOT hallucinate
- If evidence is weak or insufficient, say so
- Summarize patterns across reviews
- Highlight common positives/negatives

Input:

User Question:
{question}

Semantic query:
{semantic_question}

Filtered Context (reviews):
{retrieved_reviews}

Answer:
""".format(
    question=question,
    semantic_question=semantic_question,
    retrieved_reviews='--> '.join(results['documents'][0])
)

response = client.chat.send(
  model="openai/gpt-oss-20b:free",
  messages=[
    {"role": "user", "content": rag_context}
  ],
)

llm_response = response.choices[0].message.content

print(llm_response)

**Key Take‑aways from the reviews for a Kindle Fire HDX 8.9″ (price < $400, on sale)**  

| Category | What the reviews say |
|----------|----------------------|
| **Performance** |  The device is described as “three times faster” than its predecessor. No reports of lagging, slowdown, or software crashes were mentioned. |
| **Battery life** |  Battery “lasts slightly longer” compared to the earlier Fire HD. No complaints about rapid drain or short runtime. No reviewer cited battery problems. |
| **Display quality** |  Consistently praised as “absolutely stunning.” Extra note that the newer model has “NO blue haze” at 8.9″. No reports of dimness, color inaccuracies, or pixel‑related issues. |
| **Durability/Build quality** |  “Build quality is excellent” and the chassis is described as sturdy and lightweight. No mentions of cracks, loose hinges, or other physical wear. |

**Summary**

Across all the snippets, the Kindle Fire HDX 8.9″ receives uniformly positive feedback for its speed, l

In [ ]:
print(question)

Compare the reviews of Kindle Fire HDX 8.9" that are priced below $400 and on sale
